# Global Supply Chain Risk Prediction — 2026
**Author:** Vinay Naikv  
**Dataset:** `global_supply_chain_risk_2026.csv`  
**Objective:** Predict whether a shipment will experience a disruption (`Disruption_Occurred`) using machine learning models, and uncover the key risk drivers across global trade lanes.

---

## 1. Setup & Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
import xgboost as xgb

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded successfully.')

## 2. Load Dataset

In [ ]:
# Update path if running locally
DATA_PATH = r'C:\Users\naikv\Downloads\archive (1)\global_supply_chain_risk_2026.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Basic Information

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Descriptive Statistics ===')
df.describe()

### 3.2 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Disruption_Occurred'].value_counts()
labels = ['No Disruption (0)', 'Disruption (1)']

axes[0].bar(labels, counts.values, color=['#4c72b0', '#dd8452'], edgecolor='white')
axes[0].set_title('Disruption Occurred — Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#4c72b0', '#dd8452'], startangle=90,
            wedgeprops=dict(edgecolor='white'))
axes[1].set_title('Class Balance')

plt.tight_layout()
plt.savefig('disruption_distribution.png', bbox_inches='tight')
plt.show()

### 3.3 Categorical Feature Analysis

In [ ]:
cat_cols = ['Origin_Port', 'Destination_Port', 'Transport_Mode',
            'Product_Category', 'Weather_Condition']

fig, axes = plt.subplots(3, 2, figsize=(16, 16))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    disruption_rate = df.groupby(col)['Disruption_Occurred'].mean().sort_values(ascending=False)
    disruption_rate.plot(kind='bar', ax=axes[idx], color='#4c72b0', edgecolor='white')
    axes[idx].set_title(f'Disruption Rate by {col}')
    axes[idx].set_ylabel('Disruption Rate')
    axes[idx].set_xlabel('')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

axes[5].remove()
plt.tight_layout()
plt.savefig('categorical_disruption_rates.png', bbox_inches='tight')
plt.show()

### 3.4 Numerical Feature Distributions

In [ ]:
num_cols = ['Distance_km', 'Weight_MT', 'Fuel_Price_Index',
            'Geopolitical_Risk_Score', 'Carrier_Reliability_Score', 'Lead_Time_Days']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, hue='Disruption_Occurred', kde=True,
                 ax=axes[idx], bins=40, palette={0: '#4c72b0', 1: '#dd8452'})
    axes[idx].set_title(f'Distribution: {col}')
    axes[idx].set_xlabel(col)

plt.tight_layout()
plt.savefig('numerical_distributions.png', bbox_inches='tight')
plt.show()

### 3.5 Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 7))
corr = df[num_cols + ['Disruption_Occurred']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap — Numerical Features')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

### 3.6 Transport Mode vs Disruption

In [ ]:
plt.figure(figsize=(9, 5))
ct = pd.crosstab(df['Transport_Mode'], df['Disruption_Occurred'], normalize='index') * 100
ct.plot(kind='bar', stacked=True, color=['#4c72b0', '#dd8452'],
        edgecolor='white', figsize=(9, 5))
plt.title('Disruption % by Transport Mode')
plt.ylabel('Percentage (%)')
plt.xlabel('Transport Mode')
plt.xticks(rotation=0)
plt.legend(['No Disruption', 'Disruption'], loc='upper right')
plt.tight_layout()
plt.savefig('transport_mode_disruption.png', bbox_inches='tight')
plt.show()

### 3.7 Time-Series: Disruptions Over Time

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['YearMonth'] = df['Date'].dt.to_period('M')

monthly = df.groupby('YearMonth')['Disruption_Occurred'].mean().reset_index()
monthly['YearMonth'] = monthly['YearMonth'].astype(str)

plt.figure(figsize=(14, 4))
plt.plot(monthly['YearMonth'], monthly['Disruption_Occurred'], marker='o',
         color='#dd8452', linewidth=1.5, markersize=4)
plt.fill_between(range(len(monthly)), monthly['Disruption_Occurred'],
                 alpha=0.15, color='#dd8452')
plt.xticks(range(0, len(monthly), 2), monthly['YearMonth'][::2], rotation=45, ha='right')
plt.title('Monthly Disruption Rate Over Time')
plt.ylabel('Disruption Rate')
plt.tight_layout()
plt.savefig('monthly_disruption_trend.png', bbox_inches='tight')
plt.show()

## 4. Feature Engineering

In [ ]:
df_ml = df.copy()

# Date features
df_ml['Month']      = df_ml['Date'].dt.month
df_ml['DayOfWeek']  = df_ml['Date'].dt.dayofweek
df_ml['Quarter']    = df_ml['Date'].dt.quarter

# Interaction features
df_ml['Risk_x_Distance']  = df_ml['Geopolitical_Risk_Score'] * df_ml['Distance_km']
df_ml['Fuel_x_Distance']  = df_ml['Fuel_Price_Index']       * df_ml['Distance_km']
df_ml['Reliability_Inv']  = 1 - df_ml['Carrier_Reliability_Score']

# Label encoding
le = LabelEncoder()
for col in ['Origin_Port', 'Destination_Port', 'Transport_Mode',
            'Product_Category', 'Weather_Condition']:
    df_ml[col + '_enc'] = le.fit_transform(df_ml[col])

# Drop non-numeric columns
drop_cols = ['Shipment_ID', 'Date', 'YearMonth',
             'Origin_Port', 'Destination_Port', 'Transport_Mode',
             'Product_Category', 'Weather_Condition']
df_ml.drop(columns=drop_cols, inplace=True)

print(f'Features after engineering: {df_ml.shape[1] - 1}')
df_ml.head()

## 5. Train / Test Split & Scaling

In [ ]:
X = df_ml.drop('Disruption_Occurred', axis=1)
y = df_ml['Disruption_Occurred']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}')
print(f'Target balance (train) — 0: {(y_train==0).sum()}  1: {(y_train==1).sum()}')

## 6. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree':        DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=42),
    'XGBoost':              xgb.XGBClassifier(n_estimators=200, learning_rate=0.1,
                                              use_label_encoder=False, eval_metric='logloss',
                                              random_state=42, verbosity=0)
}

results = []

for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_sc, y_train)
        y_pred  = model.predict(X_test_sc)
        y_proba = model.predict_proba(X_test_sc)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

    report = classification_report(y_test, y_pred, output_dict=True)
    auc    = roc_auc_score(y_test, y_proba)

    results.append({
        'Model':     name,
        'Accuracy':  report['accuracy'],
        'Precision': report['1']['precision'],
        'Recall':    report['1']['recall'],
        'F1-Score':  report['1']['f1-score'],
        'ROC-AUC':   auc
    })
    print(f'{name} — Accuracy: {report["accuracy"]:.4f}  AUC: {auc:.4f}')

results_df = pd.DataFrame(results).set_index('Model')
results_df

### 6.1 Model Comparison Chart

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(results_df))
width = 0.15
colors = ['#4c72b0', '#dd8452', '#55a868', '#c44e52', '#8172b2']

fig, ax = plt.subplots(figsize=(14, 6))
for i, metric in enumerate(metrics):
    ax.bar(x + i * width, results_df[metric], width, label=metric, color=colors[i])

ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df.index, rotation=10)
ax.set_ylim(0, 1.1)
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

### 6.2 ROC Curves

In [ ]:
plt.figure(figsize=(9, 7))
colors_roc = ['#4c72b0', '#dd8452', '#55a868', '#c44e52', '#8172b2']

for (name, model), color in zip(models.items(), colors_roc):
    if name == 'Logistic Regression':
        y_proba = model.predict_proba(X_test_sc)[:, 1]
    else:
        y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_score = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc_score:.3f})')

plt.plot([0,1], [0,1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()

### 6.3 Confusion Matrix — Best Model (XGBoost)

In [ ]:
best_model = models['XGBoost']
y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Disruption', 'Disruption'])

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — XGBoost')
plt.tight_layout()
plt.savefig('confusion_matrix_xgboost.png', bbox_inches='tight')
plt.show()

print('\nClassification Report — XGBoost:')
print(classification_report(y_test, y_pred_best, target_names=['No Disruption', 'Disruption']))

## 7. Feature Importance

In [ ]:
# XGBoost built-in importance
feat_imp = pd.Series(
    best_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 7))
feat_imp.plot(kind='barh', ax=ax, color='#4c72b0', edgecolor='white')
ax.set_title('Top 15 Feature Importances — XGBoost')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

## 8. Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Random Forest':     models['Random Forest'],
    'Gradient Boosting': models['Gradient Boosting'],
    'XGBoost':           models['XGBoost']
}

print('5-Fold Stratified Cross-Validation — ROC-AUC')
print('-' * 50)
for name, model in cv_models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f'{name:25s} Mean: {scores.mean():.4f}  Std: {scores.std():.4f}')

## 9. Risk Profiling: High-Risk Routes

In [ ]:
route_risk = (
    df.groupby(['Origin_Port', 'Destination_Port'])['Disruption_Occurred']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'Disruption_Rate', 'count': 'Shipments'})
    .query('Shipments >= 30')
    .sort_values('Disruption_Rate', ascending=False)
    .head(15)
)

route_risk['Route'] = route_risk['Origin_Port'] + ' → ' + route_risk['Destination_Port']

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(route_risk['Route'], route_risk['Disruption_Rate'],
               color='#dd8452', edgecolor='white')
ax.set_xlabel('Disruption Rate')
ax.set_title('Top 15 Highest-Risk Trade Routes (min 30 shipments)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig('high_risk_routes.png', bbox_inches='tight')
plt.show()

route_risk[['Route', 'Disruption_Rate', 'Shipments']]

## 10. Key Insights Summary

In [ ]:
best_row = results_df.loc['XGBoost']
print('=== KEY INSIGHTS ===')
print(f'Total Shipments:          {len(df):,}')
print(f'Overall Disruption Rate:  {df["Disruption_Occurred"].mean()*100:.1f}%')
print()
print('Best Model: XGBoost')
print(f'  Accuracy:  {best_row["Accuracy"]*100:.2f}%')
print(f'  ROC-AUC:   {best_row["ROC-AUC"]:.4f}')
print(f'  F1-Score:  {best_row["F1-Score"]:.4f}')
print()
print('Top Risk Factors (by XGBoost feature importance):')
top_feats = pd.Series(best_model.feature_importances_,
                      index=X_train.columns).sort_values(ascending=False).head(5)
for feat, score in top_feats.items():
    print(f'  {feat:35s}: {score:.4f}')

## 11. Save Model

In [ ]:
import joblib
joblib.dump(best_model, 'xgboost_supply_chain_model.pkl')
joblib.dump(scaler,     'standard_scaler.pkl')
print('Model and scaler saved successfully.')